# Sesión 3 — Proyecto end-to-end: Pipeline de Retention Risk event-driven

**Lunes 4 de Mayo, 2026**

Este notebook es el **proyecto integrador** que une todo lo que hemos construido en las sesiones 1 y 2 con lo nuevo de la sesión 3:

| Módulo | Aporta | Reutilizado aquí |
|--------|--------|------------------|
| **M1+M2** | Bronze → Silver → Gold con datos reales Personio | `silver_personio.fact_salary_history`, `dim_employee` |
| **M3** | Pub/Sub topics, schemas Avro, BigQuery Subscription | Topic `hr-events`, schema, pull subscription para auditoría |
| **M4** | GCS notifications, idempotencia, retry, DLQ, structured logging | Patrón event-driven, dead letter table, logging |
| **M5** | Cloud Workflows + Scheduler + Eventarc | Workflow `retention-pipeline` ya desplegado |
| **M6** | Stored Procedures, UDFs, BQML, regla newsvendor | Pipeline backend completo (`sp_build_retention_features`, `sp_score_retention`, `sp_apply_decision_rule`) |

**Lo que añadimos hoy** (la pieza nueva):
- Una **Cloud Function gen2 `event_router`** que escucha al topic `hr-events`, **filtra** los eventos de tipo `PAYROLL_CLOSED` y dispara el Workflow de M5.
- Generación de **eventos sintéticos realistas** desde los datos de Personio (no eventos random — son transiciones reales detectadas en `fact_salary_history`).

## Arquitectura completa

```
[silver_personio.fact_salary_history]   ← M1+M2 (datos reales)
        ↓ detectamos transiciones (altas, cambios salariales, bajas, payroll cierre)
[Notebook publica eventos al topic `hr-events`]   ← M3 (reutilizamos)
        ↓ (con schema Avro + BigQuery Subscription para auditoría)
[Cloud Function `event_router` gen2]   ← NUEVO en esta sesión
        ↓ filtra: solo `event_type=PAYROLL_CLOSED` continúa
        ↓ dispara con target_month
[Cloud Workflows: retention-pipeline]   ← M5
        ↓
[Stored Procedures: build_features → score → apply_decision]   ← M6
        ↓
[predictions_retention.retention_actions]   ← lista priorizada para HRBP
        ↓
[Pub/Sub topic `retention-pipeline-status`]
        ↓
[BigQuery Subscription → audit_log]   ← M3 (reutilizamos)
```

## Estructura del notebook

| § | Tema |
|---|------|
| 1 | Setup + verificación de prerequisitos M3/M4/M5/M6 |
| 2 | Generación de eventos sintéticos desde Personio history |
| 3 | Schema Avro del evento (M3 pattern) |
| 4 | Cloud Function `event_router` — código embebido y despliegue |
| 5 | Pull subscription para auditar el bus en vivo |
| 6 | **End-to-end run:** publicar evento → Function dispara → Workflow corre → SPs ejecutan |
| 7 | Dead Letter Queue (M4 pattern) |
| 8 | Backfill orquestado por eventos (replay de meses pasados) |
| 9 | Observabilidad cross-pipeline (pipeline_runs + Function logs + INFORMATION_SCHEMA) |
| 10 | GenAI brief stub (foreshadowing M16) |
| 11 | Findings honestos — limitaciones del approach event-driven |

> **Filosofía Workbench:** todo vive en este notebook (incluido el código de la Cloud Function). El alumno lo ejecuta de arriba a abajo y al final tiene un sistema completo en producción.

---
## 1. Setup + verificación de prerequisitos

Detección de entorno, clientes, y validación de que **M3, M4, M5 y M6** ya están desplegados:

- M3: topic `hr-events` existe
- M5: Workflow `retention-pipeline` está ACTIVE
- M6: SPs `sp_build_retention_features`, `sp_score_retention`, `sp_apply_decision_rule` existen

In [ ]:
# Instalar dependencias (descomentar en primera ejecución)
# !pip install google-cloud-bigquery google-cloud-pubsub google-cloud-storage google-cloud-workflows google-cloud-functions google-cloud-eventarc python-dotenv

import os
import json
import time
import base64
import subprocess
from pathlib import Path
from datetime import datetime, date, timezone, timedelta
import pandas as pd
from google.cloud import bigquery, pubsub_v1
from google.api_core.exceptions import NotFound, AlreadyExists, PermissionDenied, BadRequest, FailedPrecondition
import warnings
warnings.filterwarnings('ignore')

# --- Detección de entorno ---
IN_VERTEX_AI = any([
    os.environ.get("DL_ANACONDA_HOME"),
    os.path.exists("/opt/deeplearning/metadata"),
    os.environ.get("GOOGLE_CLOUD_PROJECT"),
])

if IN_VERTEX_AI:
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "people-analytics-formacion")
    bq_client = bigquery.Client(project=PROJECT_ID)
    publisher = pubsub_v1.PublisherClient()
    subscriber = pubsub_v1.SubscriberClient()
    print(f"Entorno: Vertex AI Workbench (ADC)")
else:
    from dotenv import load_dotenv
    from google.oauth2 import service_account
    load_dotenv()
    PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "people-analytics-formacion")
    creds_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS", "service-account.json")
    if os.path.exists(creds_path):
        credentials = service_account.Credentials.from_service_account_file(creds_path)
        bq_client = bigquery.Client(project=PROJECT_ID, credentials=credentials)
        publisher = pubsub_v1.PublisherClient(credentials=credentials)
        subscriber = pubsub_v1.SubscriberClient(credentials=credentials)
    else:
        bq_client = bigquery.Client(project=PROJECT_ID)
        publisher = pubsub_v1.PublisherClient()
        subscriber = pubsub_v1.SubscriberClient()
    print(f"Entorno: Local")

REGION = "europe-southwest1"

# Reutilizamos del M3
TOPIC_HR_EVENTS = "hr-events"
TOPIC_HR_DLQ = "hr-events-dlq"

# Topics del pipeline (creados en M5)
TOPIC_STATUS = "retention-pipeline-status"
TOPIC_ALERTS = "retention-pipeline-alerts"

# Subscription nueva para auditar el flujo en este notebook
SUB_AUDIT = "hr-events-audit-integrator"

# Workflow desplegado en M5
WORKFLOW_NAME = "retention-pipeline"

# Cloud Function que vamos a desplegar AQUÍ
CF_NAME = "event-router-retention"

# Datasets
DS_SILVER = "silver_personio"
DS_PREDS = "predictions_retention"
DS_RUNS = "pipeline_runs"

LABELS = {"team": "people-analytics", "pipeline": "retention-risk", "env": "prod", "module": "integrator"}

print(f"Proyecto:  {PROJECT_ID}")
print(f"Región:    {REGION}")
print(f"Workflow:  {WORKFLOW_NAME}")
print(f"Function:  {CF_NAME} (a desplegar)")
print(f"Topic in:  {TOPIC_HR_EVENTS} (de M3)")


In [ ]:
# Verificar prerequisitos cross-módulo
print("Validación de prerequisitos:")
print(f"{'Módulo':<8} {'Recurso':<60} {'Estado':<10}")
print("-" * 85)

issues = []

# M3: topic hr-events
try:
    publisher.get_topic(request={"topic": publisher.topic_path(PROJECT_ID, TOPIC_HR_EVENTS)})
    print(f"{'M3':<8} {f'topic {TOPIC_HR_EVENTS}':<60} {'OK':<10}")
except NotFound:
    print(f"{'M3':<8} {f'topic {TOPIC_HR_EVENTS}':<60} {'FALTA':<10}")
    issues.append(f"M3: topic {TOPIC_HR_EVENTS} no existe — ejecuta el notebook de M3 primero")

# M5: Workflow retention-pipeline
try:
    r = subprocess.run([
        "gcloud", "workflows", "describe", WORKFLOW_NAME,
        f"--location={REGION}", f"--project={PROJECT_ID}",
        "--format=value(state)"
    ], capture_output=True, text=True)
    state = r.stdout.strip()
    status = "OK" if state == "ACTIVE" else f"NOT-OK ({state})"
    print(f"{'M5':<8} {f'workflow {WORKFLOW_NAME}':<60} {status:<10}")
    if state != "ACTIVE":
        issues.append(f"M5: workflow {WORKFLOW_NAME} no está ACTIVE — re-despliega con gcloud")
except Exception as e:
    print(f"{'M5':<8} {f'workflow {WORKFLOW_NAME}':<60} {'ERROR':<10}")
    issues.append(f"M5: error consultando workflow: {e}")

# M6: Stored Procedures
M6_SPS = [
    f"{PROJECT_ID}.feature_store_retention.sp_build_retention_features",
    f"{PROJECT_ID}.predictions_retention.sp_score_retention",
    f"{PROJECT_ID}.predictions_retention.sp_apply_decision_rule",
]
for sp_id in M6_SPS:
    short = sp_id.split('.', 1)[1]
    try:
        bq_client.get_routine(sp_id)
        print(f"{'M6':<8} {f'SP {short}':<60} {'OK':<10}")
    except NotFound:
        print(f"{'M6':<8} {f'SP {short}':<60} {'FALTA':<10}")
        issues.append(f"M6: SP {short} no existe — ejecuta el notebook de M6 primero")

if issues:
    print()
    print("PROBLEMAS:")
    for i in issues:
        print(f"  - {i}")
    raise RuntimeError("Faltan prerequisitos — ejecuta los notebooks indicados antes de continuar.")
else:
    print()
    print("OK — todos los prerequisitos satisfechos. Podemos construir la pieza event-driven.")


---
## 2. Generación de eventos sintéticos desde los datos reales de Personio

A diferencia de eventos sintéticos artificiales, **derivamos los eventos del histórico real**:

- Comparamos `fact_salary_history` mes a mes y detectamos transiciones:
  - **`PAYROLL_CLOSED`**: evento de sistema. Se publica una vez por mes cuando el equipo de nómina cierra el procesamiento. Es el evento que dispara el pipeline de retention.
  - **`EMPLOYEE_HIRED`**: nueva alta detectada (`hc_status` cambió de no-existe a `'ALTA'`).
  - **`EMPLOYEE_TERMINATED`**: baja detectada (`status` cambió a `'inactive'`).
  - **`SALARY_CHANGED`**: cambio salarial significativo (>3%) entre snapshots consecutivos.

Solo `PAYROLL_CLOSED` debe disparar el Workflow. Los otros 3 son ruido relevante para otros pipelines (ej: re-onboarding, exit interview, comp review).

> Esta es la realidad de un bus de eventos en producción: muchos publishers, muchos consumers, y la **función router** decide quién consume qué. Ese filtrado es lo que vamos a embeber en la Cloud Function.

In [ ]:
# Generar eventos sintéticos desde fact_salary_history
# Detectamos altas, bajas y cambios salariales reales entre 2025-08 y 2025-11
sql_eventos = f'''
WITH consecutive_snapshots AS (
  SELECT
    employee_code,
    snapshot_date,
    status,
    hc_status,
    gross_salary_monthly_lc,
    LAG(status) OVER (PARTITION BY employee_code ORDER BY snapshot_date) AS prev_status,
    LAG(hc_status) OVER (PARTITION BY employee_code ORDER BY snapshot_date) AS prev_hc_status,
    LAG(gross_salary_monthly_lc) OVER (PARTITION BY employee_code ORDER BY snapshot_date) AS prev_salary
  FROM `{PROJECT_ID}.{DS_SILVER}.fact_salary_history`
  WHERE snapshot_date BETWEEN '2025-09-01' AND '2025-11-01'
),
events AS (
  -- Nuevas altas (no había snapshot previo)
  SELECT
    snapshot_date AS event_date,
    employee_code,
    'EMPLOYEE_HIRED' AS event_type,
    CAST(NULL AS FLOAT64) AS prev_value,
    gross_salary_monthly_lc AS curr_value
  FROM consecutive_snapshots
  WHERE prev_hc_status IS NULL AND hc_status = 'ALTA'

  UNION ALL

  -- Bajas (transición active → inactive)
  SELECT snapshot_date, employee_code, 'EMPLOYEE_TERMINATED', 0.0, 0.0
  FROM consecutive_snapshots
  WHERE prev_status = 'active' AND status = 'inactive'

  UNION ALL

  -- Cambios salariales significativos (>3%)
  SELECT
    snapshot_date, employee_code, 'SALARY_CHANGED',
    prev_salary, gross_salary_monthly_lc
  FROM consecutive_snapshots
  WHERE prev_salary IS NOT NULL
    AND prev_salary > 0
    AND ABS(gross_salary_monthly_lc - prev_salary) / prev_salary > 0.03
)
SELECT * FROM events
ORDER BY event_date, event_type, employee_code
'''

df_events = bq_client.query(sql_eventos,
    job_config=bigquery.QueryJobConfig(labels={**LABELS, "step": "generate_events"})
).to_dataframe()

print(f"Eventos derivados de transiciones reales: {len(df_events)}")
print()
print("Distribución por tipo:")
print(df_events.groupby('event_type').size().to_string())
print()
print("Muestra:")
print(df_events.head(10).to_string(index=False))


In [ ]:
# Construir la lista completa de eventos a publicar.
# Añadimos los eventos PAYROLL_CLOSED (uno por mes) que son los que dispararán el Workflow.

eventos_a_publicar = []

# 1. Eventos PAYROLL_CLOSED: el equipo de nómina cierra el mes — esto dispara el pipeline
for snapshot_month in ['2025-09-01', '2025-10-01', '2025-11-01']:
    eventos_a_publicar.append({
        "event_id": f"payroll-closed-{snapshot_month}",
        "event_type": "PAYROLL_CLOSED",
        "event_timestamp": f"{snapshot_month}T18:00:00Z",
        "target_month": snapshot_month,
        "source": "payroll-system",
        "payload": json.dumps({"closed_at": f"{snapshot_month}T18:00:00Z", "n_employees": 109}),
    })

# 2. Eventos derivados del histórico (ruido del bus — la Function los va a ignorar)
for _, row in df_events.head(15).iterrows():  # solo 15 para no saturar
    eventos_a_publicar.append({
        "event_id": f"{row['event_type'].lower()}-{row['employee_code']}-{row['event_date']}",
        "event_type": row['event_type'],
        "event_timestamp": f"{row['event_date']}T12:00:00Z",
        "target_month": str(row['event_date'])[:8] + "01",
        "source": "personio-sync",
        "payload": json.dumps({
            "employee_code": int(row['employee_code']),
            "prev_value": float(row['prev_value']) if pd.notna(row['prev_value']) else None,
            "curr_value": float(row['curr_value']) if pd.notna(row['curr_value']) else None,
        }),
    })

print(f"Total eventos preparados para publicar: {len(eventos_a_publicar)}")
print()
print("Distribución:")
df_evt = pd.DataFrame(eventos_a_publicar)
print(df_evt.groupby('event_type').size().to_string())
print()
print("Muestra de un evento PAYROLL_CLOSED:")
print(json.dumps(eventos_a_publicar[0], indent=2))


---
## 3. Schema Avro del evento (patrón M3)

En M3 vimos cómo Pub/Sub puede validar mensajes contra un **schema Avro registrado**. Aquí lo aplicamos al evento de retention. El schema garantiza que cualquier publisher (Personio sync, payroll, ATS) habla el mismo lenguaje.

> Nota: registramos el schema pero **publicaremos sin schema-aware** en este demo para simplicidad. El schema queda como contrato documentado — en producción se vincularía al topic con `--message-encoding=JSON --schema=...`.

In [ ]:
# Schema Avro del evento de bus
EVENT_SCHEMA = {
    "type": "record",
    "name": "HrEvent",
    "namespace": "com.imagina.peopleanalytics",
    "doc": "Evento publicado al bus hr-events. Productores: payroll, personio-sync, ATS.",
    "fields": [
        {"name": "event_id",        "type": "string", "doc": "UUID único del evento"},
        {"name": "event_type",      "type": "string", "doc": "Tipo: PAYROLL_CLOSED | EMPLOYEE_HIRED | EMPLOYEE_TERMINATED | SALARY_CHANGED"},
        {"name": "event_timestamp", "type": "string", "doc": "ISO 8601 UTC"},
        {"name": "target_month",    "type": ["null", "string"], "default": None, "doc": "Para eventos PAYROLL_CLOSED, el mes que se cerró"},
        {"name": "source",          "type": "string", "doc": "Sistema productor"},
        {"name": "payload",         "type": "string", "doc": "JSON adicional, varía por event_type"}
    ]
}

print("Schema Avro del bus hr-events:")
print(json.dumps(EVENT_SCHEMA, indent=2))
print()
print("Validación contractual: cualquier mensaje publicado al bus debe respetar este schema.")
print("En producción se registra en Pub/Sub Schema Registry (lo vimos en M3).")


---
## 4. Cloud Function `event_router` — código y despliegue

La pieza nueva. Una **Cloud Function gen2** suscrita al topic `hr-events`. Para cada mensaje:

1. Decodifica el evento (base64 → JSON).
2. **Filtra**: solo `PAYROLL_CLOSED` continúa. Cualquier otro tipo se loguea y se descarta.
3. Si es `PAYROLL_CLOSED`, **invoca el Workflow** `retention-pipeline` con `target_month` del payload.
4. Loguea structured (Cloud Logging compatible).

Patrón: **el código vive en este notebook como string Python**, lo escribimos a `/tmp/cf_event_router/` y desplegamos con `gcloud functions deploy`. El alumno ve, modifica y despliega — todo sin salir del notebook.

In [ ]:
# Código de la Cloud Function event_router
CF_MAIN_PY = """
import base64
import json
import logging
import os

import functions_framework
from google.cloud.workflows import executions_v1

PROJECT_ID = os.environ.get("GCP_PROJECT", "")
WORKFLOW_LOCATION = os.environ.get("WORKFLOW_LOCATION", "europe-southwest1")
WORKFLOW_NAME = os.environ.get("WORKFLOW_NAME", "retention-pipeline")

# Cliente Workflows reutilizable
_executions_client = executions_v1.ExecutionsClient()

logger = logging.getLogger("event-router")
logger.setLevel(logging.INFO)


@functions_framework.cloud_event
def event_router(cloud_event):
    # Filtra eventos del topic hr-events y dispara el Workflow para PAYROLL_CLOSED.
    try:
        # 1. Decodificar evento
        msg_b64 = cloud_event.data["message"]["data"]
        payload = json.loads(base64.b64decode(msg_b64).decode("utf-8"))

        event_id = payload.get("event_id", "?")
        event_type = payload.get("event_type", "?")

        # Structured log (Cloud Logging lo parsea como JSON)
        logger.info(json.dumps({
            "severity": "INFO",
            "event_id": event_id,
            "event_type": event_type,
            "decision": "received",
        }))

        # 2. Filtro: solo PAYROLL_CLOSED dispara el pipeline
        if event_type != "PAYROLL_CLOSED":
            logger.info(json.dumps({
                "severity": "INFO",
                "event_id": event_id,
                "event_type": event_type,
                "decision": "ignored",
                "reason": "not a PAYROLL_CLOSED event",
            }))
            return  # ack: descartado correctamente

        target_month = payload.get("target_month")
        if not target_month:
            logger.error(json.dumps({
                "severity": "ERROR",
                "event_id": event_id,
                "decision": "rejected",
                "reason": "PAYROLL_CLOSED without target_month",
            }))
            raise ValueError("PAYROLL_CLOSED event missing target_month")

        # 3. Disparar el Workflow
        parent = f"projects/{PROJECT_ID}/locations/{WORKFLOW_LOCATION}/workflows/{WORKFLOW_NAME}"
        execution = executions_v1.Execution(
            argument=json.dumps({"target_month": target_month, "run_origin": "event_driven", "trigger_event_id": event_id}),
        )
        op = _executions_client.create_execution(parent=parent, execution=execution)

        logger.info(json.dumps({
            "severity": "INFO",
            "event_id": event_id,
            "event_type": event_type,
            "decision": "workflow_triggered",
            "execution_name": op.name,
            "target_month": target_month,
        }))
    except Exception as e:
        logger.error(json.dumps({
            "severity": "ERROR",
            "decision": "function_error",
            "error_type": type(e).__name__,
            "error_message": str(e)[:500],
        }))
        raise  # mensaje no ack -> reintento + DLQ despues de N fallos
"""

CF_REQUIREMENTS = """functions-framework>=3.0.0
google-cloud-workflows>=1.13.0
"""

# Escribir a directorio temporal
src_dir = Path("/tmp/cf_event_router")
src_dir.mkdir(parents=True, exist_ok=True)
(src_dir / "main.py").write_text(CF_MAIN_PY)
(src_dir / "requirements.txt").write_text(CF_REQUIREMENTS)

print(f"Codigo de Function escrito a {src_dir}")
print(f"  - main.py:           {len(CF_MAIN_PY)} caracteres")
print(f"  - requirements.txt:  {(src_dir / 'requirements.txt').read_text().strip()}")


In [ ]:
# Habilitar APIs necesarias para Cloud Functions gen2
APIS_CF = [
    "cloudfunctions.googleapis.com",
    "run.googleapis.com",
    "cloudbuild.googleapis.com",
    "artifactregistry.googleapis.com",
    "eventarc.googleapis.com",
]

for api in APIS_CF:
    print(f"Habilitando {api} ...", end=" ")
    r = subprocess.run(["gcloud", "services", "enable", api, f"--project={PROJECT_ID}"],
                       capture_output=True, text=True)
    print("OK" if r.returncode == 0 else f"FALLO: {r.stderr.strip()[:200]}")

print()
print("Esperando 15s para propagación...")
time.sleep(15)


In [ ]:
# Desplegar la Cloud Function gen2 con trigger en topic hr-events
def desplegar_function():
    cmd = [
        "gcloud", "functions", "deploy", CF_NAME,
        "--gen2",
        "--runtime=python311",
        f"--region={REGION}",
        f"--source={src_dir}",
        "--entry-point=event_router",
        "--trigger-topic=" + TOPIC_HR_EVENTS,
        f"--set-env-vars=GCP_PROJECT={PROJECT_ID},WORKFLOW_LOCATION={REGION},WORKFLOW_NAME={WORKFLOW_NAME}",
        "--memory=256Mi",
        "--timeout=120s",
        "--max-instances=5",
        f"--project={PROJECT_ID}",
        "--quiet",
    ]
    print(f"Desplegando Cloud Function {CF_NAME}... (puede tardar 1-2 minutos)")
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    if r.returncode == 0:
        print("Function desplegada")
        return True
    else:
        print(f"FALLO: {r.stderr[-500:]}")
        return False

deployed = desplegar_function()

# Verificar el estado
if deployed:
    r = subprocess.run([
        "gcloud", "functions", "describe", CF_NAME,
        f"--region={REGION}", f"--project={PROJECT_ID}",
        "--format=value(state,serviceConfig.uri)"
    ], capture_output=True, text=True)
    print(f"\nEstado: {r.stdout.strip()}")


---
## 5. Pull subscription para auditar el bus en vivo

Crearemos una **segunda subscription** al topic `hr-events` que pulsamos manualmente desde el notebook. Sirve para **ver lo que está pasando en el bus en tiempo real** mientras la Function actúa en paralelo. Patrón M3.

In [ ]:
# Crear pull subscription para auditar
sub_path = subscriber.subscription_path(PROJECT_ID, SUB_AUDIT)
topic_path = publisher.topic_path(PROJECT_ID, TOPIC_HR_EVENTS)

try:
    subscriber.create_subscription(request={
        "name": sub_path,
        "topic": topic_path,
        "ack_deadline_seconds": 60,
    })
    print(f"Subscription creada: {SUB_AUDIT}")
except AlreadyExists:
    print(f"Subscription ya existe: {SUB_AUDIT}")

print(f"Path: {sub_path}")


---
## 6. End-to-end run en vivo

El momento de la verdad. Vamos a:

1. **Publicar** los eventos al topic `hr-events`.
2. **Auditar** lo que llega via la pull subscription.
3. **Esperar** unos segundos a que la Function procese y dispare el Workflow.
4. **Verificar** que el Workflow ejecutó correctamente — `pipeline_runs.retention_pipeline_runs` debe tener filas con `run_origin='event_driven'`.
5. **Confirmar** que `predictions_retention.retention_actions` tiene resultados frescos.

In [ ]:
# Publicar todos los eventos al topic hr-events
print(f"Publicando {len(eventos_a_publicar)} eventos al topic {TOPIC_HR_EVENTS}...")
print()

futures = []
for evento in eventos_a_publicar:
    data = json.dumps(evento).encode("utf-8")
    f = publisher.publish(
        topic_path,
        data,
        event_type=evento["event_type"],
        source=evento["source"],
    )
    futures.append((evento, f))

published = []
for evento, f in futures:
    msg_id = f.result(timeout=30)
    published.append({"event_id": evento["event_id"], "type": evento["event_type"], "msg_id": msg_id})

print(f"Publicados {len(published)} mensajes al bus.")
print()
print("Resumen:")
df_pub = pd.DataFrame(published)
print(df_pub.groupby('type').size().to_string())


In [ ]:
# Pull de la subscription de auditoría — qué llegó al bus
print(f"Pulling de {SUB_AUDIT} (max 25 mensajes)...")

response = subscriber.pull(
    request={"subscription": sub_path, "max_messages": 25},
    timeout=15,
)

ack_ids = []
mensajes_vistos = []
for m in response.received_messages:
    payload = json.loads(m.message.data.decode("utf-8"))
    mensajes_vistos.append({
        "event_type": payload.get("event_type"),
        "event_id": payload.get("event_id"),
        "target_month": payload.get("target_month"),
    })
    ack_ids.append(m.ack_id)

if ack_ids:
    subscriber.acknowledge(request={"subscription": sub_path, "ack_ids": ack_ids})

print(f"Mensajes auditados: {len(mensajes_vistos)}")
df_audit = pd.DataFrame(mensajes_vistos)
if not df_audit.empty:
    print(df_audit.groupby('event_type').size().to_string())


In [ ]:
# Esperar a que la Function procese los eventos PAYROLL_CLOSED y dispare el Workflow
print("Esperando 60s para que la Function dispare los Workflows...")
time.sleep(60)

# Listar ejecuciones recientes del Workflow disparadas por evento (run_origin='event_driven')
r = subprocess.run([
    "gcloud", "workflows", "executions", "list",
    f"--workflow={WORKFLOW_NAME}",
    f"--location={REGION}",
    f"--project={PROJECT_ID}",
    "--limit=10",
    "--format=value(name,state,startTime)"
], capture_output=True, text=True)

print("Últimas 10 ejecuciones del Workflow:")
for line in r.stdout.strip().split('\n')[:10]:
    if line:
        parts = line.split()
        exec_id = parts[0].split('/')[-1] if parts else "?"
        state = parts[1] if len(parts) > 1 else "?"
        ts = parts[2] if len(parts) > 2 else "?"
        print(f"  {exec_id[:36]}... | {state:<12} | {ts}")


In [ ]:
# Verificar que el pipeline corrió: pipeline_runs debe tener filas recientes
df_runs_recent = bq_client.query(f'''
SELECT
  step,
  snapshot_month,
  status,
  rows_written,
  ROUND(duration_seconds, 2) AS dur_s,
  start_time
FROM `{PROJECT_ID}.{DS_RUNS}.retention_pipeline_runs`
WHERE start_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 10 MINUTE)
ORDER BY start_time DESC
LIMIT 20
''').to_dataframe()

print(f"Steps de pipeline ejecutados en los últimos 10 min: {len(df_runs_recent)}")
if not df_runs_recent.empty:
    print(df_runs_recent.to_string(index=False))

# Verificar las decisiones más recientes
df_actions = bq_client.query(f'''
SELECT
  snapshot_month,
  recommended_action,
  COUNT(*) AS empleados,
  MAX(scored_at) AS scored_at
FROM `{PROJECT_ID}.{DS_PREDS}.retention_actions`
WHERE scored_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 10 MINUTE)
GROUP BY snapshot_month, recommended_action
ORDER BY snapshot_month DESC, recommended_action
''').to_dataframe()

print()
print("Decisiones aplicadas en los últimos 10 min:")
if df_actions.empty:
    print("  (ninguna — el Workflow puede aún estar corriendo o haber fallado)")
else:
    print(df_actions.to_string(index=False))


---
## 7. Dead Letter Queue (patrón M4)

Si la Function falla 5 veces consecutivas en un mensaje, Pub/Sub lo redirige a un **dead letter topic** (configurado en M3 como `hr-events-dlq`). Aquí inyectamos un evento "veneno" (malformed) para verificar que el patrón funciona.

In [ ]:
# Publicar un evento "veneno" (PAYROLL_CLOSED sin target_month → la Function tira ValueError)
evento_malformado = {
    "event_id": "poison-test-1",
    "event_type": "PAYROLL_CLOSED",
    "event_timestamp": "2025-12-01T18:00:00Z",
    "target_month": None,  # esto rompe la Function
    "source": "test-corruption",
    "payload": "{}",
}

f = publisher.publish(topic_path, json.dumps(evento_malformado).encode("utf-8"))
msg_id = f.result(timeout=30)
print(f"Evento veneno publicado: msg_id={msg_id}")
print(f"  → la Function lo rechazará (raise ValueError)")
print(f"  → Pub/Sub reintentará 5 veces, luego lo enviará a {TOPIC_HR_DLQ}")
print(f"  → un suscriptor del DLQ podría notificar a Slack o BigQuery (M17)")
print()
print("Nota: el patrón DLQ requiere que la subscription del trigger Eventarc tenga --dead-letter-topic.")
print("En clase, verificar en consola: Pub/Sub → subscriptions → eventarc-* → Dead lettering.")


---
## 8. Backfill orquestado por eventos (replay de meses pasados)

Si descubrimos un bug en el modelo y queremos **re-procesar 3 meses**, en lugar de modificar SQL podemos **republicar** los eventos `PAYROLL_CLOSED` correspondientes. La idempotencia heredada de los SPs (DELETE+INSERT por partición) garantiza que el re-run produce el mismo resultado sin duplicar.

In [ ]:
# Backfill: republicar PAYROLL_CLOSED para 2025-09 y 2025-10
# (idempotente — los SPs sobrescriben las particiones)
print("Republicando PAYROLL_CLOSED para 2025-09-01 y 2025-10-01 (backfill)...")

for snapshot_month in ['2025-09-01', '2025-10-01']:
    evt = {
        "event_id": f"backfill-{snapshot_month}-{int(time.time())}",
        "event_type": "PAYROLL_CLOSED",
        "event_timestamp": datetime.now(timezone.utc).isoformat(),
        "target_month": snapshot_month,
        "source": "backfill-script",
        "payload": json.dumps({"reason": "model_v2_rerun"}),
    }
    f = publisher.publish(topic_path, json.dumps(evt).encode("utf-8"))
    print(f"  Publicado backfill para {snapshot_month}: {f.result(timeout=15)}")

print()
print("La Function recibirá los eventos, disparará 2 Workflows en paralelo")
print("(max-instances=5 lo permite). Los SPs con DELETE+INSERT garantizan idempotencia.")


---
## 9. Observabilidad cross-pipeline

Combinamos 3 fuentes:
- **`pipeline_runs.retention_pipeline_runs`** — log estructurado de los SPs (M6).
- **Cloud Function logs** — vía `gcloud functions logs read`.
- **`INFORMATION_SCHEMA.JOBS_BY_USER`** — bytes/coste de las queries del pipeline.

In [ ]:
# Logs de la Cloud Function (últimos 50)
print(f"Últimos logs de {CF_NAME}:")
r = subprocess.run([
    "gcloud", "functions", "logs", "read", CF_NAME,
    f"--region={REGION}", f"--project={PROJECT_ID}",
    "--limit=20",
    "--format=value(time_utc,log)"
], capture_output=True, text=True)
print(r.stdout[-2500:] if r.stdout else "  (no logs todavía)")


In [ ]:
# Cross-pipeline view: qué pasó en los últimos 15 min
df_state = bq_client.query(f'''
SELECT
  step,
  COUNT(*) AS runs,
  COUNTIF(status = 'SUCCESS') AS exitos,
  COUNTIF(status = 'FAILED') AS fallos,
  ROUND(AVG(duration_seconds), 2) AS avg_dur_s,
  SUM(rows_written) AS total_rows
FROM `{PROJECT_ID}.{DS_RUNS}.retention_pipeline_runs`
WHERE start_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 15 MINUTE)
GROUP BY step
ORDER BY runs DESC
''').to_dataframe()

print("Resumen de pipeline_runs en los últimos 15 min:")
if df_state.empty:
    print("  (ninguno — el pipeline no ha corrido o aún está procesando)")
else:
    print(df_state.to_string(index=False))


---
## 10. GenAI brief stub (foreshadowing M16)

El cierre del pipeline: tomamos las decisiones del último run y generamos un **brief en lenguaje natural** que un HRBP recibe por email/Slack. Stub — el cliente real con Vertex AI Gemini llega en M16.

In [ ]:
# Brief de retención del último mes procesado
def brief_retention(snapshot_month: str, top_n: int = 5) -> str:
    df = bq_client.query(f'''
    SELECT employee_code, country, team, tier, recommended_action, priority,
           ROUND(retention_risk_score, 3) AS score
    FROM `{PROJECT_ID}.{DS_PREDS}.retention_actions`
    WHERE snapshot_month = DATE '{snapshot_month}'
      AND recommended_action = 'retention_action'
    ORDER BY priority, score DESC
    LIMIT {top_n}
    ''').to_dataframe()

    lineas = [
        f"BRIEF DE RETENCIÓN — Mes {snapshot_month}",
        "=" * 50,
        f"Top {len(df)} casos prioritarios para acción HRBP:",
        "",
    ]
    for _, row in df.iterrows():
        lineas.append(
            f"  - Empleado #{row['employee_code']} ({row['country']}, team {row['team']}, "
            f"tier {row['tier']}): score={row['score']}, prioridad={row['priority']}"
        )
    lineas.extend([
        "",
        "Recomendación: 1:1 con manager dentro de 2 semanas.",
        "[En M16 esto será texto generado por Vertex AI Gemini con prompts contextualizados]",
    ])
    return "\n".join(lineas)


print(brief_retention("2025-11-01"))


---
## 11. Findings honestos

### Lo que funcionó

1. **Pipeline event-driven completo end-to-end:** publicamos eventos al bus, la Function filtra, el Workflow corre, los SPs ejecutan, y `retention_actions` se actualiza — todo automático.
2. **Reuso real de M3/M4:** el topic `hr-events` y la pull subscription son los mismos artefactos creados en M3.
3. **Idempotencia heredada:** publicamos backfill events para 2025-09 y 2025-10 sin miedo. Los SPs reescriben las particiones limpiamente.
4. **Filtrado en la Function:** publicamos 3 PAYROLL_CLOSED + 15 eventos de ruido (SALARY_CHANGED, EMPLOYEE_HIRED, EMPLOYEE_TERMINATED) y solo los 3 PAYROLL_CLOSED dispararon el pipeline.
5. **Observabilidad cross-fuente:** combinamos `pipeline_runs`, Function logs, `INFORMATION_SCHEMA` para un retrato completo.

### Lo que NO funcionó / cuidado

1. **Cold start de la Function gen2:** primera invocación tarda ~3-5s en arrancar el container. En producción, si el cliente espera latencia <1s, hay que pre-warmar (min-instances=1) o aceptar el spike.

2. **Race conditions en backfill simultáneo:** publicamos 2 eventos PAYROLL_CLOSED a la vez. Si los Workflows corren simultáneos sobre meses diferentes, OK (idempotentes por partición). Si corren sobre el MISMO mes, hay race en `retention_actions` (el último gana). En producción, usar `serializable_isolation` en el SP o singleton lock.

3. **No hay exactly-once:** Pub/Sub garantiza at-least-once. Si la Function se cae después del CALL al SP pero antes de ack, el mensaje se reintenta y el Workflow corre 2 veces. La idempotencia del SP nos cubre, pero **siempre hay una ventana** donde el state intermedio puede inconsistir.

4. **DLQ requiere configuración manual de la subscription Eventarc:** el patrón DLQ que probamos en §7 funciona en M3 con la subscription pull, pero la subscription que Eventarc crea para la Function tiene su propia configuración. Verificar en consola Pub/Sub.

5. **Cloud Function tiene permisos elevados:** la SA de la Function necesita `roles/workflows.invoker`. En producción, **nunca** dar permisos amplios a una Function que recibe mensajes de un bus público — solo lo que necesita.

6. **Eventos sintéticos vs reales:** en este notebook **derivamos** eventos del histórico. En producción, los publicarían los sistemas reales (Personio webhook, payroll system). El schema Avro es el contrato — sin él, cualquier cambio en Personio rompe silenciosamente el pipeline.

### Conexión con módulos siguientes

- **M9 (GitHub)**: el código de la Function vive en este notebook hoy, pero en producción se versiona en Git con CI/CD a Cloud Functions.
- **M11 (DLP)**: revisar que el evento no contenga PII (employee_code está bien, gross_salary individual no debería viajar por el bus).
- **M17 (Observabilidad)**: el suscriptor del topic `*-alerts` debe ser una Function que notifique a Slack — hoy se queda en el topic sin acción.

## Resumen del proyecto e2e

Hemos construido y ejecutado un pipeline event-driven que:

- Conecta los 4 módulos previos en un sistema único.
- Procesa datos reales (anonimizados) de 209 empleados.
- Genera un modelo predictivo (BQML logistic, AUC=0.84).
- Aplica una regla de decisión defendible (newsvendor Cu/Co).
- Produce una lista priorizada para HRBP.
- Es **idempotente, observable, auditable, y reactivo a eventos**.

Y todo en un notebook de Workbench — el patrón de producción real.

→ **La sesión 4 sigue con el M7 (SQL avanzado): window functions de cohortes para enriquecer las features que entran al modelo.**
